In [2]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
import numpy as np

# =========================
# CONFIG
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
DATA_DIR = "./content/dataset"

# =========================
# DATA PIPELINE
# =========================
def build_datasets(data_dir):
    train_ds = keras.utils.image_dataset_from_directory(
        os.path.join(data_dir, "train"),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary"
    )

    val_ds = keras.utils.image_dataset_from_directory(
        os.path.join(data_dir, "validation"),
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary"
    )

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = (
        train_ds
        .map(lambda x, y: (x / 255.0, y))
        .cache()
        .shuffle(1000)
        .prefetch(buffer_size=AUTOTUNE)
    )

    val_ds = (
        val_ds
        .map(lambda x, y: (x / 255.0, y))
        .cache()
        .prefetch(buffer_size=AUTOTUNE)
    )

    return train_ds, val_ds


# =========================
# MODEL (TRANSFER LEARNING)
# =========================
def build_model():
    base_model = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False  # freeze backbone

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )

    return model


# =========================
# TRAINING
# =========================
def train(model, train_ds, val_ds):
    callbacks = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=3),
        keras.callbacks.ModelCheckpoint("best_model.keras", save_best_only=True)
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    return history


# =========================
# INFERENCE
# =========================
class CatDogPredictor:
    def __init__(self, model_path):
        self.model = keras.models.load_model(model_path)
        self.img_size = IMG_SIZE

    def preprocess(self, image_path):
        img = cv2.imread(image_path)
        img = cv2.resize(img, self.img_size)
        img = img / 255.0
        return np.expand_dims(img, axis=0)

    def predict(self, image_path):
        x = self.preprocess(image_path)
        prob = self.model.predict(x)[0][0]

        label = "dog" if prob > 0.5 else "cat"
        confidence = float(prob if prob > 0.5 else 1 - prob)

        return {
            "label": label,
            "confidence": round(confidence, 4)
        }


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    train_ds, val_ds = build_datasets(DATA_DIR)

    model = build_model()
    model.summary()

    train(model, train_ds, val_ds)

    predictor = CatDogPredictor("best_model.keras")

    print(predictor.predict("./content/dataset/test/dogs/dog (10).jpg"))
    print(predictor.predict("./content/dataset/test/cats/cat (10).jpg"))

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,201 (9.26 MB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 98ms/step - accuracy: 0.9577 - auc: 0.9923 - loss: 0.1091 - val_accuracy: 0.9834 - val_auc: 0.9978 - val_loss: 0.0467 - learning_rate: 1.0000e-04
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 78s 124ms/step - accuracy: 0.9783 - auc: 0.9971 - loss: 0.0604 - val_accuracy: 0.9856 - val_auc: 0.9979 - val_loss: 0.0432 - learning_rate: 1.0000e-04
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 107s 171ms/step - accuracy: 0.9824 - auc: 0.9983 - loss: 0.0454 - val_accuracy: 0.9840 - val_auc: 0.9983 - val_loss: 0.0420 - learning_rate: 1.0000e-04
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 66s 105ms/step - accuracy: 0.9854 - auc: 0.9988 - loss: 0.0376 - val_accuracy: 0.9848 - val_auc: 0.9983 - val_loss: 0.0420 - learning_rate: 1.0000e-04
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 71s 113ms/step - accuracy: 0.9890 - auc: 0.9990 - loss: 0.0320 - val_accuracy: 0.9840 - val_auc: 0.9980 - val_loss: 0.0419 - learning_rate: 1.0000e-04
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 1004s